# Cold Google Drive read · Pika H3 Flex

From VS Code, provision a **new standard CPU** Colab server and run the cell. It mounts Drive read-only, verifies the five files from the Pika baseline and copies their full contents sequentially to the empty VM disk. The timer starts after authorization and metadata checks.

Each file is read in 16 MiB blocks, matching the copy in the GPU benchmark. The output includes per-file and overall transfer rates using decimal GB and MB/s. The temporary copies are removed after measurement.

In [ ]:
import json
import shutil
import socket
import time
from pathlib import Path
from tempfile import TemporaryDirectory

from google.colab import drive

FILES = {'diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors': 20970379616,
 'loras/pika_h3_flex_fl2va_v1_bf16.safetensors': 1956172304,
 'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors': 15687142551,
 'vae/minimax_h3_video_vae_int8_convrot.safetensors': 2811065184,
 'vae/minimax_h3_audio_vae_fp32.safetensors': 605254808}

source_root = Path("/content/drive/MyDrive/colab-models/Pika-H3-Flex")
runtime_root = Path("/content")
marker = runtime_root / ".pika-h3-cold-drive-read-attempt"
total_bytes = sum(FILES.values())
hostname = socket.gethostname()
boot_id = (Path("/proc/sys/kernel/random/boot_id").read_text().strip())
vm_id = f"{hostname}:{boot_id}"
print(f"Colab VM: {hostname}. Mounting Drive read-only.", flush=True)
drive.mount("/content/drive", timeout_ms=300_000, readonly=True)

baseline = json.loads((source_root / "baseline.json").read_text())
if baseline["vm_id"] == vm_id:
    raise RuntimeError("Use a fresh VM distinct from the original Pika benchmark.")
if baseline["model_bytes"] != total_bytes:
    raise RuntimeError("The checkpoint size differs from the original benchmark.")

sources = source_root / "models"
for name, expected in FILES.items():
    path = sources / name
    if path.stat().st_size != expected:
        raise IOError(f"Unexpected source size: {name}")
if shutil.disk_usage(runtime_root).free < total_bytes:
    raise RuntimeError("The fresh VM needs 42.03 GB of free local disk space.")
if marker.exists():
    raise RuntimeError("This runtime already attempted a read. Use a new VM for a cold run.")
marker.touch(exist_ok=False)

measurements = []
with TemporaryDirectory(prefix="pika-h3-drive-read-", dir=runtime_root) as temp:
    target = Path(temp)
    print(f"Starting a cold sequential copy of {total_bytes / 1e9:.2f} GB.", flush=True)
    started = time.perf_counter()
    for name, expected in FILES.items():
        short_name = Path(name).name
        print(f"Reading {short_name} ({expected / 1e9:.2f} GB)", flush=True)
        destination = target / short_name
        file_started = time.perf_counter()
        copied = 0
        next_report = 4 * 2**30
        with (sources / name).open("rb") as reader, destination.open("wb") as writer:
            while block := reader.read(16 * 2**20):
                writer.write(block)
                copied += len(block)
                if copied >= next_report:
                    print(f"  {copied / 2**30:.0f} GiB copied", flush=True)
                    next_report += 4 * 2**30
        elapsed = time.perf_counter() - file_started
        if copied != expected or destination.stat().st_size != expected:
            raise IOError(f"Incomplete copy: {short_name}")
        rate = copied / elapsed / 1e6
        measurements.append({"file": short_name, "bytes": copied, "seconds": round(elapsed, 3),
                             "MB_per_second": round(rate, 2)})
        print(f"  Finished in {elapsed:.1f} s at {rate:.1f} MB/s", flush=True)
    total_seconds = time.perf_counter() - started
    total_rate = total_bytes / total_seconds / 1e6
    print(f"Cold Drive-to-disk: {total_bytes / 1e9:.2f} GB in {total_seconds:.1f} s"
          f" ({total_rate:.1f} MB/s)", flush=True)
    print(json.dumps({
        "vm": hostname, "total_bytes": total_bytes, "total_seconds": round(total_seconds, 3),
        "MB_per_second": round(total_rate, 2), "files": measurements,
    }, indent=2), flush=True)